#### This Jupyter Notebook will demonstrate how to perform an in silico digestion of a protein sequence using the `pyteomics` package.

### Step 1: Importing Required Libraries

We need the following Python libraries:
- `Bio.SeqIO` from Biopython to read the FASTA file.
- `pyteomics.parser` to perform tryptic digestion.
- `csv` to save the results.

In [1]:
from Bio import SeqIO
from pyteomics import parser
import pandas
from deeplc import DeepLC


### Step 2: Parsing the FASTA File

We will read the FASTA file containing protein sequences and store the peptide fragments in different data structures.


Here we will use the pyteomics package to perform an in silico trypsin digest

In [2]:
pep_dict = {}
for record in SeqIO.parse("methanococcus_maripaludis_proteome.fasta", "fasta"):
    protein_id = str(record.id)
    #print("Protein:", protein_id)
    sequence = str(record.seq)
    # use pyteomics to generate tryptic peptides
    tryptic_peptides = parser.cleave(sequence, "trypsin")
    for pep in tryptic_peptides:
        #print(pep)
        if pep in pep_dict:
            pep_dict[pep].append(protein_id)
        else:
            pep_dict[pep] = [protein_id]
#print(pep_dict)

In [3]:
# Calculate the percentage of non-proteotypic peptides (peptides found in more than one protein)
non_proteotypic = [pep for pep, prots in pep_dict.items() if len(prots) > 1]
num_non_proteotypic = len(non_proteotypic)
total_peptides = len(pep_dict)

percent_non_proteotypic = (num_non_proteotypic / total_peptides) * 100
print(f"Percentage of non-proteotypic peptides: {percent_non_proteotypic:.2f}%")

Percentage of non-proteotypic peptides: 5.52%


In [4]:
import matplotlib.pyplot as plt
from collections import Counter
import numpy as np

# 1. Length distribution of all non-proteotypic peptides
non_proteotypic_lengths = [len(pep) for pep in non_proteotypic]
plt.figure(figsize=(8,4))
plt.hist(non_proteotypic_lengths, bins=range(min(non_proteotypic_lengths), max(non_proteotypic_lengths)+2), alpha=0.7, color='orange', edgecolor='black')
plt.xlabel('Peptide Length')
plt.ylabel('Count')
plt.title('Length Distribution of Non-Proteotypic Peptides')
plt.show()
plt.savefig('non_proteotypic_length_distribution.png')

# 2. Percentage of non-proteotypic peptides for each peptide length
all_lengths = [len(pep) for pep in pep_dict.keys()]
nonprot_length_counts = Counter(non_proteotypic_lengths)
all_length_counts = Counter(all_lengths)
lengths = sorted(all_length_counts.keys())
percent_nonprot_per_length = [ (nonprot_length_counts[l]/all_length_counts[l])*100 if all_length_counts[l]>0 else 0 for l in lengths ]
plt.figure(figsize=(8,4))
plt.plot(lengths, percent_nonprot_per_length, marker='o', color='blue')
plt.xlabel('Peptide Length')
plt.ylabel('% Non-Proteotypic Peptides')
plt.title('Percentage of Non-Proteotypic Peptides by Length')
plt.grid(True)
plt.show()
plt.savefig('percent_non_proteotypic_by_length.png')

In [5]:
# Predict retention times for all peptides using DeepLC
peptides = pep_dict.keys()
modifications = ['' for n in peptides]
df = pandas.DataFrame({'seq': peptides, 'modifications': modifications})
print(peptides)
print(modifications)
print(df)

dict_keys(['DLVEK', 'IK', 'GIAK', 'LVVIAQDVQPEEIVAHIPVICDEK', 'K', 'EALGK', 'FEISQELEEK', 'AVEK', 'GIAYSYSSTK', 'AAGLEVPTSAIAVVAEGSADELK', 'TAEVVANAEK', 'A', 'GANEVTK', 'MAVYVK', 'LNGLK', 'MK', 'FLEK', 'QITNSTNQTTQALADAISDATSQMSNITD', 'LTSK', 'GQIAMELGILVMAAVAVAAIAAYFYATNVSNTGK', 'LGFMQIHTLPYALK', 'LAFSK', 'AFNEIR', 'ILGVAFAFFGAFAAFGIGNMVQTNSVADAVASNFGVDPLITGFVLAIFTAAVILGGIK', 'TDHPGR', 'LYR', 'GLPDHGLGK', 'AFPGLTDLTGASLTAASFDALMPMGGLIVTIGLVFFAYSTVLGWSYYGEK', 'MDFVSLVNTVNSFVWGPYMLVLLLGTGIFLTLR', 'YAEAVLAIK', 'SIGK', 'HQDETSEGDISHFQALMTALAATIGTGNIAGVATAYVLGGPGAIFWMWVTAFFGMATK', 'R', 'QALVSMTGTFLDTIVVCTITGLVLTIAGLK', 'CFEYLIGTK', 'ATGIIVPFMAVFYILAGLVILAMNIGYIIPAFGTIFSSAFNFSAGFGALIGTAIMWGVK', 'GIR', 'YR', 'NEAK', 'TVDDNGEMAGGPMYFLEK', 'GVFSNEAGLGSAPIAAAAAK', 'IAFVLVAFWGATASLPLVWNIADTLNGAMAIPNLIGLLLLSGVVVSETK', 'NA', 'ETK', 'LKPIPDR', 'EFAR', 'EGK', 'NGR', 'NMR', 'NIMDSDEPGIILTTSGMLSGGPSVEYFK', 'EIPLMGK', 'AFPHFDAPEFSIEEIDAVVVTHAHLDHCGFIPGLFR', 'DETEK', 'TCLYHQTPESR', 'CIDLASTAYK', 'DDSWVR

In [6]:
# Initialize DeepLC and make predictions
dlc = DeepLC()
predictions = dlc.make_preds(seq_df=df, calibrate=False)

Unable to get the following position: 2
Unable to get the following position: 3
Unable to get the following position: 1
Truncating peptide (too long): TIPVVFLLMPAVILLSILLSIPFGGSISQFQFSGGFSFSTDFVPVLFLLLLAATFEELGW
Could not add the following atom: 60 ('R', None)
Could not add the following value: pos 1 for atom Se with value 1
Could not add the following atom: Se
Skipping the following (not in library): 1 ('U', None)
Could not add the following value: pos 9 for atom Se with value 1
Skipping the following (not in library): 9 ('U', None)
Truncating peptide (too long): SLFPISITGTSTSLLNFFPFFGGAVLQQVCGLVIASYGLNALGGYTSAGYQMAWLLLAVG
Could not add the following atom: 60 ('M', None)
Could not add the following atom: 61 ('V', None)
Could not add the following atom: 62 ('I', None)
Could not add the following atom: 63 ('A', None)
Could not add the following atom: 64 ('T', None)
Could not add the following atom: 65 ('I', None)
Could not add the following atom: 66 ('C', None)
Could not add the followi

37/37 ━━━━━━━━━━━━━━━━━━━━ 12s 294ms/step


In [7]:
# The predictions are retention times in minutes (0-60 min scale for C18 RP-HPLC)
# Add predictions to the dataframe
df['retention_time'] = predictions

print("Sample peptides with predicted retention times:")
print(df.head(10))
print(f"\nRetention time statistics (in minutes):")
print(f"Min: {df['retention_time'].min():.2f}")
print(f"Max: {df['retention_time'].max():.2f}")
print(f"Mean: {df['retention_time'].mean():.2f}")
print(f"Median: {df['retention_time'].median():.2f}")

# Histogram of retention times
plt.figure(figsize=(10, 5))
plt.hist(df['retention_time'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
plt.xlabel('Retention Time (minutes)')
plt.ylabel('Number of Peptides')
plt.title('Distribution of Predicted Retention Times for All Peptides (C18 RP-HPLC)')
plt.grid(True, alpha=0.3)
plt.show()
plt.savefig('retention_time_distribution.png')

Sample peptides with predicted retention times:
                        seq modifications  retention_time
0                     DLVEK                     -7.348721
1                        IK                    -36.310921
2                      GIAK                    -33.119568
3  LVVIAQDVQPEEIVAHIPVICDEK                    102.599350
4                         K                    -40.631775
5                     EALGK                    -30.233152
6                FEISQELEEK                     39.206238
7                      AVEK                    -41.330540
8                GIAYSYSSTK                      2.072954
9   AAGLEVPTSAIAVVAEGSADELK                     87.004036

Retention time statistics (in minutes):
Min: -93.74
Max: 294.11
Mean: 33.41
Median: 21.55


In [34]:
# Calculate overlapping elution peaks considering Gaussian distribution
# Peptides elute with Gaussian distribution, peak width = FWHM (Full Width at Half Maximum)
# For Gaussian: FWHM = 2.355 * σ, so σ = FWHM / 2.355

def calculate_overlapping_peaks(retention_times, peak_width_seconds, max_retention_time_min):
    """
    Calculate percentage of peptides with overlapping elution peaks.
    Two peaks overlap when their Gaussian distributions intersect significantly.
    We consider peaks to overlap when center-to-center distance < 2.5 * sigma
    (approximately where 95% confidence intervals start to overlap)
    """
    # Convert peak width from seconds to minutes
    peak_width_min = peak_width_seconds / 60.0
    
    # Calculate standard deviation from FWHM
    # FWHM = 2.355 * σ
    sigma = peak_width_min / 2.355
    
    # Threshold for overlap: 2.5 * sigma
    overlap_threshold = 2.5 * sigma
    
    overlapping_peptides = set()
    total_comparisons = 0
    
    # Compare all pairs of peptides
    for i in range(len(retention_times)):
        for j in range(i + 1, len(retention_times)):
            total_comparisons += 1
            rt_diff = abs(retention_times[i] - retention_times[j])
            
            if rt_diff < overlap_threshold:
                overlapping_peptides.add(i)
                overlapping_peptides.add(j)
    
    percent_overlapping = (len(overlapping_peptides) / len(retention_times)) * 100
    
    return percent_overlapping, len(overlapping_peptides), sigma, overlap_threshold

In [35]:
# Scenario 1: 60 min max, 30 second peak width
print("="*60)
print("SCENARIO 1: Standard C18 RP-HPLC (60 min max)")
print("="*60)
percent_overlap_60, num_overlap_60, sigma_60, threshold_60 = calculate_overlapping_peaks(
    df['retention_time'].tolist(), 
    peak_width_seconds=30, 
    max_retention_time_min=60
)
print(f"Peak width (FWHM): 30 seconds = 0.5 minutes")
print(f"Standard deviation (σ): {sigma_60:.4f} minutes = {sigma_60*60:.2f} seconds")
print(f"Overlap threshold (2.5σ): {threshold_60:.4f} minutes = {threshold_60*60:.2f} seconds")
print(f"Number of peptides with overlapping peaks: {num_overlap_60}")
print(f"Percentage of peptides with overlapping peaks: {percent_overlap_60:.2f}%")

SCENARIO 1: Standard C18 RP-HPLC (60 min max)
Peak width (FWHM): 30 seconds = 0.5 minutes
Standard deviation (σ): 0.2123 minutes = 12.74 seconds
Overlap threshold (2.5σ): 0.5308 minutes = 31.85 seconds
Number of peptides with overlapping peaks: 36878
Percentage of peptides with overlapping peaks: 99.96%


In [36]:
# Scenario 2: 120 min max, 35 second peak width
print("\n" + "="*60)
print("SCENARIO 2: Extended C18 RP-HPLC (120 min max)")
print("="*60)
percent_overlap_120, num_overlap_120, sigma_120, threshold_120 = calculate_overlapping_peaks(
    df['retention_time'].tolist(), 
    peak_width_seconds=35, 
    max_retention_time_min=120
)
print(f"Peak width (FWHM): 35 seconds = {35/60:.4f} minutes")
print(f"Standard deviation (σ): {sigma_120:.4f} minutes = {sigma_120*60:.2f} seconds")
print(f"Overlap threshold (2.5σ): {threshold_120:.4f} minutes = {threshold_120*60:.2f} seconds")
print(f"Number of peptides with overlapping peaks: {num_overlap_120}")
print(f"Percentage of peptides with overlapping peaks: {percent_overlap_120:.2f}%")


SCENARIO 2: Extended C18 RP-HPLC (120 min max)
Peak width (FWHM): 35 seconds = 0.5833 minutes
Standard deviation (σ): 0.2477 minutes = 14.86 seconds
Overlap threshold (2.5σ): 0.6192 minutes = 37.15 seconds
Number of peptides with overlapping peaks: 36881
Percentage of peptides with overlapping peaks: 99.97%


In [37]:
# Comparison
print("\n" + "="*60)
print("COMPARISON")
print("="*60)
print(f"Change in overlapping peptides: {num_overlap_120 - num_overlap_60}")
print(f"Change in percentage: {percent_overlap_120 - percent_overlap_60:.2f}%")
if percent_overlap_120 < percent_overlap_60:
    print(f"✓ Increasing retention time REDUCED overlap by {percent_overlap_60 - percent_overlap_120:.2f}%")
else:
    print(f"✓ Increasing retention time INCREASED overlap by {percent_overlap_120 - percent_overlap_60:.2f}%")


COMPARISON
Change in overlapping peptides: 3
Change in percentage: 0.01%
✓ Increasing retention time INCREASED overlap by 0.01%
